# 04 — Figures

Publication-quality figures:
1. World map of concentration index
2. Concentration curves — selected countries / all countries
3. CI vs mean productivity loss scatter
4. Regional summaries

**Prerequisite:** run notebook 03 to generate results CSVs.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import yaml

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

with open(ROOT / 'config' / 'config.yaml') as f:
    config = yaml.safe_load(f)

DATA    = ROOT / 'data'
RESULTS = ROOT / 'results'

# Adjust epoch to match what was processed
EPOCH_START = config['wbgt']['start_year']
EPOCH_END   = config['wbgt']['end_year']

ci_df     = pd.read_csv(RESULTS / f'inequality_results_{EPOCH_START}_{EPOCH_END}.csv')
curves_df = pd.read_csv(RESULTS / f'concentration_curves_{EPOCH_START}_{EPOCH_END}.csv')

print(f'Countries: {len(ci_df)}')
ci_df.head()

## Figure 1 — World map of Concentration Index

In [ ]:
world = gpd.read_file(DATA / config['data']['boundaries'])

# Identify ISO column
for col in ['ADM0_A3', 'ISO_A3', 'iso_a3', 'GID_0']:
    if col in world.columns:
        iso_col = col
        break

world_ci = world.merge(ci_df[['iso3', 'CI']], left_on=iso_col, right_on='iso3', how='left')

fig, ax = plt.subplots(figsize=(16, 8))
world.plot(ax=ax, color='lightgrey', edgecolor='white', linewidth=0.3)
world_ci.dropna(subset=['CI']).plot(
    ax=ax, column='CI', cmap='RdBu', vmin=-1, vmax=1,
    edgecolor='white', linewidth=0.3, legend=True,
    legend_kwds={'label': 'Concentration Index', 'shrink': 0.6}
)
ax.set_title(f'Heatwave Productivity Loss Inequality (CI) — {EPOCH_START}–{EPOCH_END}', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig(RESULTS / f'fig1_ci_map_{EPOCH_START}_{EPOCH_END}.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 2 — Concentration curves

In [ ]:
# All countries, coloured by CI value
norm = mcolors.TwoSlopeNorm(vcenter=0, vmin=-1, vmax=1)
cmap = plt.cm.RdBu

fig, ax = plt.subplots(figsize=(8, 8))

for iso, grp in curves_df.groupby('iso3'):
    ci_val = ci_df.loc[ci_df['iso3'] == iso, 'CI'].values
    if len(ci_val) == 0:
        continue
    color = cmap(norm(ci_val[0]))
    ax.plot(grp['cum_pop_share'], grp['cum_risk_share'],
            color=color, linewidth=0.8, alpha=0.7)

# Line of equality
ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, label='Line of equality')

sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
plt.colorbar(sm, ax=ax, label='Concentration Index')

ax.set_xlabel('Cumulative population share (poorest → wealthiest)')
ax.set_ylabel('Cumulative productivity loss share')
ax.set_title(f'Concentration Curves — {EPOCH_START}–{EPOCH_END}')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / f'fig2_concentration_curves_{EPOCH_START}_{EPOCH_END}.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 3 — CI vs Mean Productivity Loss

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

sc = ax.scatter(
    ci_df['mean_productivity_loss'],
    ci_df['CI'],
    c=ci_df['rwi_coverage'],
    cmap='YlOrRd_r',
    s=60,
    edgecolors='grey',
    linewidth=0.5,
)

# Label selected countries
highlight = ['IND', 'NGA', 'BRA', 'IDN', 'BGD', 'PAK', 'ETH', 'EGY']
for _, row in ci_df[ci_df['iso3'].isin(highlight)].iterrows():
    ax.annotate(row['iso3'],
                xy=(row['mean_productivity_loss'], row['CI']),
                xytext=(4, 4), textcoords='offset points', fontsize=8)

plt.colorbar(sc, ax=ax, label='RWI population coverage')
ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
ax.set_xlabel('Mean annual productivity loss (fraction)')
ax.set_ylabel('Concentration Index')
ax.set_title(f'Heatwave Exposure vs Inequality — {EPOCH_START}–{EPOCH_END}')
plt.tight_layout()
plt.savefig(RESULTS / f'fig3_exposure_vs_inequality_{EPOCH_START}_{EPOCH_END}.png', dpi=150, bbox_inches='tight')
plt.show()

## Figure 4 — CI distribution by region

In [ ]:
# Add region from world boundaries
region_cols = ['REGION_UN', 'SUBREGION', 'region_un', 'subregion']
region_col = next((c for c in region_cols if c in world.columns), None)

if region_col:
    region_map = world.set_index(iso_col)[region_col].to_dict()
    ci_df['region'] = ci_df['iso3'].map(region_map)

    fig, ax = plt.subplots(figsize=(12, 6))
    regions = ci_df.groupby('region')['CI'].apply(list)
    ax.boxplot(regions.values, labels=regions.index, vert=True)
    ax.axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
    ax.set_ylabel('Concentration Index')
    ax.set_title(f'CI by Region — {EPOCH_START}–{EPOCH_END}')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig(RESULTS / f'fig4_ci_by_region_{EPOCH_START}_{EPOCH_END}.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Region column not found in boundaries file — skipping.')